# 🎯 LeadGen AI — Google Colab 12GB RAM Backend Runner
Ye notebook aapke LeadGen scraper ko **12 GB RAM** par chalata hai aur Cloudflare Tunnel ke zariye aapke Frontend se connect karta hai.

### 🚀 Step 1: Dependencies & Cloudflare Tunnel Install karein (1-2 Min)
Is cell ko run karke required packages aur Chromium browser install karein.

In [ ]:
!pip install -q fastapi uvicorn playwright pandas openpyxl requests python-dotenv pydantic
!playwright install chromium
!playwright install-deps chromium

# Download and install Cloudflared Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("\n✅ All Dependencies & Cloudflared installed successfully!")

### 📥 Step 2: GitHub Repository Clone karein
Latest backend code GitHub se pull karein.

In [ ]:
!rm -rf leadgen
!git clone https://github.com/nikhilcodeworks/leadgen.git
%cd leadgen/backend
print("✅ Backend ready in:", !pwd)

### ⚡ Step 3: Backend Server & Cloudflare Tunnel Start karein
Ye cell chalane par aapko **trycloudflare.com** link milega. Use frontend me paste karein.

In [ ]:
import os
import re
import subprocess
import time

# Optional: Set HF Token directly yahan bhi kar sakte hain (ya frontend UI se pass karein)
os.environ["HF_TOKEN"] = ""
os.environ["HUGGINGFACE_API_KEY"] = os.environ.get("HF_TOKEN", "")

# 1. Start FastAPI backend on port 8000
server_log = open("server.log", "w")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
server_proc = subprocess.Popen(["python", "-u", "api_server.py"], stdout=server_log, stderr=server_log, env=env)

# 2. Start Cloudflare Tunnel to expose port 8000
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tunnel_log,
    stderr=tunnel_log
)

print("⏳ Starting Cloudflare Tunnel... Please wait 5-10 seconds...")
tunnel_url = None
for _ in range(35):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "="*65)
    print("🎉 AAPKA 12GB RAM COLAB BACKEND LIVE HAI:")
    print(f"👉 {tunnel_url}")
    print("="*65)
    print("\nIs URL ko apne LeadGen Frontend me daal kar '⚡ Test Both' karein!")
else:
    print("❌ Tunnel URL nahi mil saki. tunnel.log contents:")
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
            print(f.read())
    print("server.log contents:")
    if os.path.exists("server.log"):
        with open("server.log", "r", encoding="utf-8", errors="ignore") as f:
            print(f.read())
print("\n📡 Tip: Live logs monitor karne ke liye neeche Step 4 Cell run karein ya Frontend me \"Live Terminal\" open karein!")


### 🖥️ Step 4: Live Server & Scraper Logs Dekhein (Real-Time)
Jab aap Frontend Web UI se koi Search ya Scraping start karenge, yahan terminal par har action, listing URL, aur AI analysis live print hoga!

*(Stop karne ke liye Colab cell ke left me Stop icon click karein)*

In [ ]:
import os
import time

print('📡 Live Scraper Stream Active. Waiting for logs...\n' + '-'*65)
log_file = 'server.log'
while not os.path.exists(log_file):
    time.sleep(1)

with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
    # Print existing lines
    for line in f.readlines()[-30:]:
        print(line.rstrip())
    # Follow live
    while True:
        line = f.readline()
        if line:
            print(line.rstrip(), flush=True)
        else:
            time.sleep(0.5)
